In [7]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
#Respiratory Disease Detection Using Lung Sound Analysis
csv_path = r"c:\Users\n2101\OneDrive\Desktop\Respiratory_Disease_Detection_Using_Lung_Sound_Analysis\Respiratory_Sound_Database\patient_diagnosis.csv"

# Load CSV
patient_diagnosis = pd.read_csv(csv_path, 
                        names=['Patient Number', 'Diagnosis'])
patient_diagnosis


,Patient Number,Diagnosis
0,101,URTI
1,102,Healthy
2,103,Asthma
3,104,COPD
4,105,URTI
...,...,...
121,222,COPD
122,223,COPD
123,224,Healthy
124,225,Healthy


In [4]:
categories = patient_diagnosis['Diagnosis'].unique()
categories

array(['URTI', 'Healthy', 'Asthma', 'COPD', 'LRTI', 'Bronchiectasis',
       'Pneumonia', 'Bronchiolitis'], dtype=object)

In [8]:
csv_path = r"c:\Users\n2101\OneDrive\Desktop\Respiratory_Disease_Detection_Using_Lung_Sound_Analysis\demographic_info.txt"

demographic_info = pd.read_csv(csv_path, names=[
    'Patient Number', 'Age', 'Gender', 'Adult BMI (kg/m2)', 'Child Weight (kg)', 'Child Height (cm)'], delimiter=' ')

demographic_info

,Patient Number,Age,Gender,Adult BMI (kg/m2),Child Weight (kg),Child Height (cm)
0,101,3.00,F,NaN,19.0,99.0
1,102,0.75,F,NaN,9.8,73.0
2,103,70.00,F,33.00,NaN,NaN
3,104,70.00,F,28.47,NaN,NaN
4,105,7.00,F,NaN,32.0,135.0
...,...,...,...,...,...,...
121,222,60.00,M,NaN,NaN,NaN
122,223,NaN,NaN,NaN,NaN,NaN
123,224,10.00,F,NaN,32.3,143.0
124,225,0.83,M,NaN,7.8,74.0


In [9]:
def clean_metadata(metadata_path):
  df = pd.read_csv(metadata_path, sep=" ", header=None, names=[ 'Patient Number', 'Age', 'Gender', 'Adult BMI (kg/m2)', 'Child Weight (kg)', 'Child Height (cm)'])
  #considering child under 18
  mask_kids = df["Age"] < 18
  df.loc[mask_kids, "Child BMI (kg/m2)"] = df.loc[mask_kids].apply(
      lambda row: row["Child Weight (kg)"] / ((row["Child Height (cm)"] / 100) ** 2)
      if not pd.isnull(row["Child Weight (kg)"]) and not pd.isnull(row["Child Height (cm)"]) else np.nan,
      axis=1,
  )
  # merge bmi and drop useless
  df["BMI (kg/m2)"] = df["Adult BMI (kg/m2)"].fillna(df["Child BMI (kg/m2)"])
  df.drop(["Child BMI (kg/m2)", "Child Height (cm)", "Child Weight (kg)", "Adult BMI (kg/m2)"], axis=1, inplace=True)
  # Add column for sound path
  return df

csv_path = r"c:\Users\n2101\OneDrive\Desktop\Respiratory_Disease_Detection_Using_Lung_Sound_Analysis\demographic_info.txt"
clean_demographic_info=clean_metadata(csv_path)
clean_demographic_info

,Patient Number,Age,Gender,BMI (kg/m2)
0,101,3.00,F,19.385777
1,102,0.75,F,18.389942
2,103,70.00,F,33.000000
3,104,70.00,F,28.470000
4,105,7.00,F,17.558299
...,...,...,...,...
121,222,60.00,M,NaN
122,223,NaN,NaN,NaN
123,224,10.00,F,15.795393
124,225,0.83,M,14.243974


In [10]:
merged_df = pd.merge(clean_demographic_info, patient_diagnosis, on='Patient Number', how='inner')
merged_df

,Patient Number,Age,Gender,BMI (kg/m2),Diagnosis
0,101,3.00,F,19.385777,URTI
1,102,0.75,F,18.389942,Healthy
2,103,70.00,F,33.000000,Asthma
3,104,70.00,F,28.470000,COPD
4,105,7.00,F,17.558299,URTI
...,...,...,...,...,...
121,222,60.00,M,NaN,COPD
122,223,NaN,NaN,NaN,COPD
123,224,10.00,F,15.795393,Healthy
124,225,0.83,M,14.243974,Healthy


**Patient number** (101,102,...,226) <br>
**Recording index** <br>
**Chest location** (Trachea (Tc), {Anterior (A), Posterior (P), Lateral (L)}{left (l), right (r)}) <br>
**Acquisition mode** (sequential/single channel (sc), simultaneous/multichannel (mc)) <br>
**Recording equipment** (AKG C417L Microphone, 3M Littmann Classic II SE Stethoscope, 3M Litmmann 3200 Electronic Stethoscope, WelchAllyn Meditron Master Elite Electronic Stethoscope)

In [11]:
import os
import glob
# Now preprocessing Audio_and_Text files
# Extraction info from file names

audio_path = r"c:\Users\n2101\OneDrive\Desktop\Respiratory_Disease_Detection_Using_Lung_Sound_Analysis\Respiratory_Sound_Database\audio_and_txt_files"

# adding columns 
file_names = []
patient_number = []
recording_index = []
chest_location = []
acquisition_mode = []
recording_equipment = []


for path in glob.glob(os.path.join(audio_path, "*.wav")):
    file = os.path.basename(path).replace(".wav", "")
    parts = file.split("_")

    if len(parts) >= 5:
        # Extract components
        file_names.append(file)
        patient_number.append(parts[0])
        recording_index.append(parts[1])
        chest_location.append(parts[2])
        acquisition_mode.append(parts[3])
        recording_equipment.append(parts[4])


# Create DataFrame
df_audio = pd.DataFrame({
    "Filename": file_names,
    "Patient Number": patient_number,
    "Recording Index": recording_index,
    "Chest Location": chest_location,
    "Acquisition Mode": acquisition_mode,
    "Recording Equipment": recording_equipment
})

df_audio

,Filename,Patient Number,Recording Index,Chest Location,Acquisition Mode,Recording Equipment
0,101_1b1_Al_sc_Meditron,101,1b1,Al,sc,Meditron
1,101_1b1_Pr_sc_Meditron,101,1b1,Pr,sc,Meditron
2,102_1b1_Ar_sc_Meditron,102,1b1,Ar,sc,Meditron
3,103_2b2_Ar_mc_LittC2SE,103,2b2,Ar,mc,LittC2SE
4,104_1b1_Al_sc_Litt3200,104,1b1,Al,sc,Litt3200
...,...,...,...,...,...,...
916,225_1b1_Pl_sc_Meditron,225,1b1,Pl,sc,Meditron
917,226_1b1_Al_sc_Meditron,226,1b1,Al,sc,Meditron
918,226_1b1_Ll_sc_Meditron,226,1b1,Ll,sc,Meditron
919,226_1b1_Pl_sc_LittC2SE,226,1b1,Pl,sc,LittC2SE


In [9]:
merged_df = merged_df.dropna()
nan_counts = merged_df.isna().sum()
print("NaN values per column:")
print(nan_counts)

NaN values per column:
Patient Number    0
Age               0
Gender            0
BMI (kg/m2)       0
Diagnosis         0
dtype: int64


In [12]:
import os
import numpy as np
import librosa
import librosa.feature

# --- Audio feature extraction for each recording ---
# This function turns a .wav file into a compact fixed-length feature vector.
def extract_audio_features(file_path, sr=None, n_mfcc=5):
    audio, sr = librosa.load(file_path, sr=sr)  # keep original sampling rate if sr=None

    features = {}

    # Basic energy (Measures how strong/loud the signal is over time.)
    rms = librosa.feature.rms(y=audio)[0]
    features['rms_mean'] = float(rms.mean())
    features['rms_std'] = float(rms.std())

    # Zero-crossing rate (Counts how often the signal changes sign (crosses zero). Noisy or high‑frequency sounds cross zero more often.)
    zcr = librosa.feature.zero_crossing_rate(y=audio)[0]
    features['zcr_mean'] = float(zcr.mean())
    features['zcr_std'] = float(zcr.std())

    # Spectral centroid (how "bright" the sound is) (Roughly indicates where the “center of mass” of the spectrum is (how “bright” or high‑frequency the sound is.)
    spec_centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)[0]
    features['spec_centroid_mean'] = float(spec_centroid.mean())
    features['spec_centroid_std'] = float(spec_centroid.std())

    #  Mel-Frequency Cepstral Coefficients. MFCCs (reduced set for speed) (shape of the spectrum.)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
    for i in range(n_mfcc):
        coeff = mfcc[i]
        features[f'mfcc_{i+1}_mean'] = float(coeff.mean())
        features[f'mfcc_{i+1}_std'] = float(coeff.std())

    return features

In [13]:
# --- Build per-recording audio feature table and aggregate per patient ---

audio_features_rows = []

df_audio_tmp = df_audio.copy()
merged_df_tmp = merged_df.copy()
df_audio_tmp['Patient Number'] = df_audio_tmp['Patient Number'].astype(str)
merged_df_tmp['Patient Number'] = merged_df_tmp['Patient Number'].astype(str)

df_audio_with_meta = df_audio_tmp.merge(
    merged_df_tmp[['Patient Number', 'Age', 'Gender', 'BMI (kg/m2)', 'Diagnosis']],
    on='Patient Number',
    how='inner'
 )

print('Recording-level rows with metadata:', df_audio_with_meta.shape)

# Limit the number of recordings used for feature extraction to speed things up
MAX_RECORDINGS = 650  # you can increase later when things work

# audio_path should already point to the audio_and_txt_files folder
for _, row in df_audio_with_meta.head(MAX_RECORDINGS).iterrows():
    base_name = str(row['Filename'])
    if base_name.lower().endswith('.wav'):
        wav_name = base_name
    else:
        wav_name = base_name + '.wav'

    file_path = os.path.join(audio_path, wav_name)

    try:
        feats = extract_audio_features(file_path, sr=None)
        feats['Filename'] = base_name
        feats['Patient Number'] = row['Patient Number']
        audio_features_rows.append(feats)
    except Exception as e:
        print(f'Error processing {file_path}: {e}')

df_audio_features = pd.DataFrame(audio_features_rows)
print('Extracted audio feature table shape:', df_audio_features.shape)
df_audio_features.head()

Recording-level rows with metadata: (920, 10)
Extracted audio feature table shape: (650, 18)


,rms_mean,rms_std,zcr_mean,zcr_std,spec_centroid_mean,spec_centroid_std,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,mfcc_3_mean,mfcc_3_std,mfcc_4_mean,mfcc_4_std,mfcc_5_mean,mfcc_5_std,Filename,Patient Number
0,0.053982,0.026954,0.001192,0.001074,291.511393,195.038017,-536.597229,21.693485,80.337196,22.293802,62.652004,14.370365,44.194176,11.147983,31.550388,8.163524,101_1b1_Al_sc_Meditron,101
1,0.033046,0.015870,0.001184,0.001359,398.452034,218.887595,-599.051575,18.912687,81.443336,19.288376,57.341728,11.173090,34.708996,9.030241,23.990013,7.452848,101_1b1_Pr_sc_Meditron,101
2,0.021396,0.010526,0.001393,0.001684,479.680354,255.093297,-609.898743,18.624407,95.075272,22.405935,62.562664,16.282793,32.625652,11.752246,20.025837,7.689635,102_1b1_Ar_sc_Meditron,102
3,0.245882,0.134919,0.000636,0.000846,303.540148,239.480481,-429.030975,14.253368,59.045887,12.769360,46.305904,7.108954,34.043316,7.537869,27.111345,7.195107,103_2b2_Ar_mc_LittC2SE,103
4,0.075943,0.041452,0.036546,0.017768,160.261320,70.730581,-322.799377,75.152802,193.387131,34.458702,22.684046,31.214394,-5.406460,14.893450,17.699913,7.441942,104_1b1_Al_sc_Litt3200,104


In [14]:
# --- Aggregate audio features per patient and rebuild modeling dataset ---

if df_audio_features.empty:
    raise RuntimeError('df_audio_features is empty; run the extraction cell first.')

# Identify only the purely numeric audio feature columns
audio_feature_cols = [
    c for c in df_audio_features.columns
    if c not in ['Filename', 'Patient Number']
]

df_audio_patient = df_audio_features.groupby('Patient Number')[audio_feature_cols].mean().reset_index()
print('Per-patient aggregated audio features shape:', df_audio_patient.shape)

# Merge with demographic + diagnosis info (one row per patient)
merged_df_audio = merged_df_tmp.merge(df_audio_patient, on='Patient Number', how='inner')
print('Merged patient+audio dataset shape:', merged_df_audio.shape)
merged_df_audio.head()

Per-patient aggregated audio features shape: (85, 17)
Merged patient+audio dataset shape: (85, 21)


,Patient Number,Age,Gender,BMI (kg/m2),Diagnosis,rms_mean,rms_std,zcr_mean,zcr_std,spec_centroid_mean,...,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,mfcc_3_mean,mfcc_3_std,mfcc_4_mean,mfcc_4_std,mfcc_5_mean,mfcc_5_std
0,101,3.00,F,19.385777,URTI,0.043514,0.021412,0.001188,0.001216,344.981713,...,-567.824402,20.303086,80.890266,20.791089,59.996866,12.771728,39.451586,10.089112,27.770201,7.808186
1,102,0.75,F,18.389942,Healthy,0.021396,0.010526,0.001393,0.001684,479.680354,...,-609.898743,18.624407,95.075272,22.405935,62.562664,16.282793,32.625652,11.752246,20.025837,7.689635
2,103,70.00,F,33.000000,Asthma,0.245882,0.134919,0.000636,0.000846,303.540148,...,-429.030975,14.253368,59.045887,12.769360,46.305904,7.108954,34.043316,7.537869,27.111345,7.195107
3,104,70.00,F,28.470000,COPD,0.064422,0.037743,0.028613,0.014307,138.839089,...,-353.854151,72.715888,173.181564,36.630589,37.514763,30.147767,3.743239,17.596910,14.784420,10.929361
4,105,7.00,F,17.558299,URTI,0.028077,0.014889,0.042457,0.023536,3121.908330,...,-380.818878,54.609409,153.423965,49.389408,71.694679,10.961445,-37.766163,25.298254,-35.287979,18.080275


In [36]:
# --- Redefine features to include audio and retrain RandomForest (with tuning) ---
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Numerical = demographics + aggregated audio features
basic_num_cols = ['Age', 'BMI (kg/m2)']
audio_num_cols = [c for c in audio_feature_cols]
numerical_cols = basic_num_cols + audio_num_cols
categorical_cols = ['Gender']

X = merged_df_audio[numerical_cols + categorical_cols]
y = merged_df_audio['Diagnosis']

# Show class counts before any filtering
print("Class counts (all classes) for audio+demographics model:\n", y.value_counts())
print("Full audio+demographics dataset shape:", X.shape)

# Optional: filter out ultra-rare classes so stratification and CV work better
min_samples_per_class = 3
class_counts = y.value_counts()
classes_to_keep = class_counts[class_counts >= min_samples_per_class].index

X = X[y.isin(classes_to_keep)]
y = y[y.isin(classes_to_keep)]

print(f"\nAfter filtering to classes with >= {min_samples_per_class} samples:")
print(y.value_counts())
print("Filtered audio+demographics dataset shape:", X.shape)

# Train/test split WITH stratification on the filtered labels
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

# Prepare processed features once for tuning
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

# Base RandomForest and small hyperparameter grid for tuning
rf_base = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    class_weight='balanced_subsample',
    n_jobs=-1
)

param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [None, 10, 20],
    'min_samples_leaf': [1, 2]
}

grid_search = GridSearchCV(
    rf_base,
    param_grid,
    cv=5,
    n_jobs=-1,
    scoring='accuracy'
 )

grid_search.fit(X_train_proc, y_train)

rf_clf = grid_search.best_estimator_

print("\nBest hyperparameters:", grid_search.best_params_)
print("Best CV accuracy:", grid_search.best_score_)

# Evaluate on the held-out test set
y_pred = rf_clf.predict(X_test_proc)

print('\nTest accuracy with tuned RF (filtered classes):', accuracy_score(y_test, y_pred))
# If you want more detail, you can uncomment the following lines:
# print('\nClassification report:\n', classification_report(y_test, y_pred))
# print('\nConfusion matrix:\n', confusion_matrix(y_test, y_pred))

Class counts (all classes) for audio+demographics model:
 Diagnosis
COPD              44
Healthy           17
URTI              10
Bronchiectasis     4
Bronchiolitis      4
Pneumonia          3
LRTI               2
Asthma             1
Name: count, dtype: int64
Full audio+demographics dataset shape: (85, 19)

After filtering to classes with >= 3 samples:
Diagnosis
COPD              44
Healthy           17
URTI              10
Bronchiectasis     4
Bronchiolitis      4
Pneumonia          3
Name: count, dtype: int64
Filtered audio+demographics dataset shape: (82, 19)

Best hyperparameters: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 200}
Best CV accuracy: 0.6923076923076923

Test accuracy with tuned RF (filtered classes): 0.7647058823529411


In [169]:
import numpy as np
import pandas as pd
import os

# --- Predict on a recording that WAS used during training ---

# 1) Build a merged recording+metadata table for ALL recordings (same as in the unused-recording cell)
df_audio_tmp = df_audio.copy()
merged_df_tmp = merged_df.copy()

df_audio_tmp['Patient Number'] = df_audio_tmp['Patient Number'].astype(str)
merged_df_tmp['Patient Number'] = merged_df_tmp['Patient Number'].astype(str)

df_audio_with_meta_full = df_audio_tmp.merge(
    merged_df_tmp[['Patient Number', 'Age', 'Gender', 'BMI (kg/m2)', 'Diagnosis']],
    on='Patient Number',
    how='inner'
 )

# 2) Identify recordings that WERE used for feature extraction (training set)
used_files = set(df_audio_features['Filename'].astype(str).unique())

# 3) Filter to recordings that WERE used before
df_used_candidates = df_audio_with_meta_full[
    df_audio_with_meta_full['Filename'].astype(str).isin(used_files)
 ]

if df_used_candidates.empty:
    raise RuntimeError("No used recordings found. Make sure df_audio_features is built.")

# Take one used recording at random from the used set
sample_row_used = df_used_candidates.sample(29).iloc[0]

# 4) Build the .wav path for this recording
base_name_used = str(sample_row_used['Filename'])
if base_name_used.lower().endswith('.wav'):
    wav_name_used = base_name_used
else:
    wav_name_used = base_name_used + '.wav'

file_path_used = os.path.join(audio_path, wav_name_used)

# 5) Extract audio features for THIS recording only
feats_used = extract_audio_features(file_path_used, sr=None)

# 6) Build a single-row DataFrame with all model input features
feature_dict_used = {}

# Demographic numerical features
feature_dict_used['Age'] = float(sample_row_used['Age'])
feature_dict_used['BMI (kg/m2)'] = float(sample_row_used['BMI (kg/m2)'])

# Categorical feature
feature_dict_used['Gender'] = sample_row_used['Gender']

# Audio features (same columns as used during training)
for col in audio_feature_cols:
    if col in feats_used:
        feature_dict_used[col] = float(feats_used[col])
    else:
        feature_dict_used[col] = np.nan

X_used = pd.DataFrame([feature_dict_used])

# 7) Transform with the SAME preprocessor and predict with the trained model
X_used_processed = preprocessor.transform(X_used)
if hasattr(X_used_processed, 'toarray'):
    X_used_processed = X_used_processed.toarray()

pred_label_used = rf_clf.predict(X_used_processed)[0]

print("TRAINING-used recording filename:", base_name_used)
print("Patient Number:", sample_row_used['Patient Number'])
print("True diagnosis (from CSV):", sample_row_used['Diagnosis'])
print("Predicted diagnosis (model):", pred_label_used)
print("Patient Age, BMI, Gender:", sample_row_used['Age'], sample_row_used['BMI (kg/m2)'], sample_row_used['Gender'])

TRAINING-used recording filename: 144_1b1_Al_sc_Meditron
Patient Number: 144
True diagnosis (from CSV): Healthy
Predicted diagnosis (model): Healthy
Patient Age, BMI, Gender: 3.0 16.7 M


In [ ]:
import numpy as np

import pandas as pd

import os

# --- Predict on a NEW recording that was NOT used during training ---

# 1) Build a merged recording+metadata table for ALL recordings

df_audio_tmp = df_audio.copy()

merged_df_tmp = merged_df.copy()

df_audio_tmp['Patient Number'] = df_audio_tmp['Patient Number'].astype(str)

merged_df_tmp['Patient Number'] = merged_df_tmp['Patient Number'].astype(str)



df_audio_with_meta_full = df_audio_tmp.merge(

    merged_df_tmp[['Patient Number', 'Age', 'Gender', 'BMI (kg/m2)', 'Diagnosis']],

    on='Patient Number',

    how='inner'

)

# 2) Identify recordings that were ALREADY used for feature extraction (training set)

used_files = set(df_audio_features['Filename'].astype(str).unique())

# 3) Filter to recordings that were NOT used before

df_new_candidates = df_audio_with_meta_full[

    ~df_audio_with_meta_full['Filename'].astype(str).isin(used_files)

]

if df_new_candidates.empty:

    raise RuntimeError("No unused recordings found. Try increasing MAX_RECORDINGS or rerun extraction with fewer recordings.")

# Take one new recording at random from the unused set

sample_row = df_new_candidates.sample(10).iloc[1]

# 4) Build the .wav path for this recording

base_name = str(sample_row['Filename'])

if base_name.lower().endswith('.wav'):

    wav_name = base_name

else:

    wav_name = base_name + '.wav'

file_path = os.path.join(audio_path, wav_name)

# 5) Extract audio features for THIS recording only

feats = extract_audio_features(file_path, sr=None)

# 6) Build a single-row DataFrame with all model input features

feature_dict = {}

# Demographic numerical features

feature_dict['Age'] = float(sample_row['Age'])

feature_dict['BMI (kg/m2)'] = float(sample_row['BMI (kg/m2)'])

# Categorical feature

feature_dict['Gender'] = sample_row['Gender']

# Audio features (same columns as used during training)

for col in audio_feature_cols:

    if col in feats:

        feature_dict[col] = float(feats[col])

    else:

        feature_dict[col] = np.nan

X_new = pd.DataFrame([feature_dict])

# 7) Transform with the SAME preprocessor and predict with the trained model

X_new_processed = preprocessor.transform(X_new)

if hasattr(X_new_processed, 'toarray'):

    X_new_processed = X_new_processed.toarray()

pred_label = rf_clf.predict(X_new_processed)[0]


print("New (unseen) recording filename:", base_name)

print("Patient Number:", sample_row['Patient Number'])

print("True diagnosis (from CSV):", sample_row['Diagnosis'])

print("Predicted diagnosis (model):", pred_label)

print("Patient Age, BMI, Gender:", sample_row['Age'], sample_row['BMI (kg/m2)'], sample_row['Gender'])

New (unseen) recording filename: 191_2b1_Pl_mc_LittC2SE
Patient Number: 191
True diagnosis (from CSV): Pneumonia
Predicted diagnosis (model): COPD
Patient Age, BMI, Gender: 74.0 36.0 F


In [26]:
import numpy as np
import pandas as pd
import os

# --- Predict on a COMPLETELY EXTERNAL recording (not in the dataset) ---

# 1) Set the path to your external .wav file (outside the dataset)
custom_file_path = r"c:\\Users\\n2101\\OneDrive\\Desktop\\Respiratory_Disease_Detection_Using_Lung_Sound_Analysis\\Respiratory_Sound_Database\\audio_and_txt_files\\227_1b1_Pl_sc_LittC2SE.wav"  # TODO: change this

# 2) Manually enter demographic info for this person
custom_age = 60.0      # e.g. 60 years
custom_bmi = 24.0      # e.g. BMI 24.0
custom_gender = "M"    # "M" or "F"

# 3) Extract audio features for this external recording
feats_ext = extract_audio_features(custom_file_path, sr=None)

# 4) Build a single-row DataFrame with all model input features
feature_dict_ext = {}

# Demographic numerical features
feature_dict_ext['Age'] = float(custom_age)
feature_dict_ext['BMI (kg/m2)'] = float(custom_bmi)

# Categorical feature
feature_dict_ext['Gender'] = custom_gender

# Audio features (same columns as used during training)
for col in audio_feature_cols:
    if col in feats_ext:
        feature_dict_ext[col] = float(feats_ext[col])
    else:
        feature_dict_ext[col] = np.nan

X_ext = pd.DataFrame([feature_dict_ext])

# 5) Transform with the SAME preprocessor and predict with the trained model
X_ext_processed = preprocessor.transform(X_ext)
if hasattr(X_ext_processed, 'toarray'):
    X_ext_processed = X_ext_processed.toarray()

pred_label_ext = rf_clf.predict(X_ext_processed)[0]

print("Custom external recording file:", custom_file_path)
print("Predicted diagnosis (model):", pred_label_ext)
print("Provided Age, BMI, Gender:", custom_age, custom_bmi, custom_gender)

Custom external recording file: c:\\Users\\n2101\\OneDrive\\Desktop\\Respiratory_Disease_Detection_Using_Lung_Sound_Analysis\\Respiratory_Sound_Database\\audio_and_txt_files\\227_1b1_Pl_sc_LittC2SE.wav
Predicted diagnosis (model): COPD
Provided Age, BMI, Gender: 60.0 24.0 M
